In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-06-24 20:13:47--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  4.30MB/s    in 0.2s    

2026-06-24 20:13:48 (4.30 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [5]:
with open('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars) # number of unique characters possible the model can predict
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
#Now tokenize encode and decoder
# create a mapping from characters to integers
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #encode takes a string and vector store 
decode = lambda l: ''.join([itos[i] for i in l]) # decode: take a list outputs a string

print(encode("hello"))
print(decode(encode("hello")))


[46, 43, 50, 50, 53]
hello


In [9]:
%pip install torch
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # translation of first 1000 characters into vector form



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  

In [ ]:
# splitting data into train and validataion sets 
n = int(0.60*len(data))
n2 = int(0.80 * len(data))

train_data = data[:n]
val_data = data[n:n2]



In [11]:
block_size = 8
train_data[:block_size +1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target=y[t]
    print(f" when input is {context} the target is: {target}")

 when input is tensor([18]) the target is: 47
 when input is tensor([18, 47]) the target is: 56
 when input is tensor([18, 47, 56]) the target is: 57
 when input is tensor([18, 47, 56, 57]) the target is: 58
 when input is tensor([18, 47, 56, 57, 58]) the target is: 1
 when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
 when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
 when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [ ]:
torch.manual_seed(1337)
batch_size =4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x= torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('---')
# 32 indepenedent in one batch 
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context} the target is: {target}")




inputs:
torch.Size([4, 8])
tensor([[39, 57,  1, 58, 46, 47, 57,  1],
        [17, 31, 32, 17, 30, 10,  0, 15],
        [58, 46, 43, 56,  6,  1, 32, 63],
        [53, 59,  1, 39, 56, 58,  1, 57]])
targets:
torch.Size([4, 8])
tensor([[57,  1, 58, 46, 47, 57,  1, 44],
        [31, 32, 17, 30, 10,  0, 15, 53],
        [46, 43, 56,  6,  1, 32, 63, 56],
        [59,  1, 39, 56, 58,  1, 57, 61]])
---
when input is tensor([39]) the target is: 57
when input is tensor([39, 57]) the target is: 1
when input is tensor([39, 57,  1]) the target is: 58
when input is tensor([39, 57,  1, 58]) the target is: 46
when input is tensor([39, 57,  1, 58, 46]) the target is: 47
when input is tensor([39, 57,  1, 58, 46, 47]) the target is: 57
when input is tensor([39, 57,  1, 58, 46, 47, 57]) the target is: 1
when input is tensor([39, 57,  1, 58, 46, 47, 57,  1]) the target is: 44
when input is tensor([17]) the target is: 31
when input is tensor([17, 31]) the target is: 32
when input is tensor([17, 31, 32]) the 

In [ ]:
# Bigram Neural model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self,vocab_size):
        super().__init__()
        # each toekn directly reads off the logits for the next token from the vocabulary
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx,targets= None):
        
        logits = self.token_embedding_table(idx) # B,T,C: prediciting identity by current token
        
        if targets is None:
            loss = None
            
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C) 
            targets = targets.view(B*T)
            
            loss = F.cross_entropy(logits, targets) # how well is logits getting to target?
        return logits, loss

    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # nn.Module directly calls forward
            logits = logits[:,-1,:] # only focus on the last char
            probs = F.softmax(logits, dim=-1) # apply softmax regression 
            idx_next = torch.multinomial(probs,num_samples=1) #(B,1)
            idx = torch.cat((idx,idx_next),dim=-1)
        
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb,yb)
print(logits.shape)
print(loss)
idx = torch.zeros((1,1), dtype =torch.long)
print(decode(m.generate(idx, max_new_tokens =100)[0].tolist()))




torch.Size([800, 65])
tensor(4.7067, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
# pytorch optimizer AdamW
optimizer = torch.optim.AdamW(m.parameters() ,lr=1e-3) #3e^-4



In [64]:
batch_size = 100
for _ in range(10000):
    
    xb,yb= get_batch('train')
    
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() # recompute gradients
    
    optimizer.step()
    
print(loss.item(), end="")
print(decode(m.generate(idx= torch.zeros((1,1),dtype=torch.long), max_new_tokens =300)[0].tolist()))

2.509934425354004
As f whan wes.
WAn an II wambaige, whrurisat mprdw tlif y g amy th Yoo'd thfathe,
ENCangint MAn?
'MENChingee be; yofe watar cy!
Thandsher, stwis, mpl zer Be m y wan KI hawing ththe
Myedartaveakn the sse!
OR IUENathe PJuche t ge hy the?
Nor lston LOrotes at,
F pe
Ty ber? l fot--hredonf kinis,
CO::
CI


In [68]:
##Math trick in Self Attention##
# torch for how matrix multiplication used for weights aggregation

torch.manual_seed(32)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim = True)
b = torch.randint(0,10,(3,2)).float()
c= a  @ b
print('a=')
print(a)

print('b=')
print(b)

print('c=')
print(c)




a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[5., 3.],
        [3., 4.],
        [4., 6.]])
c=
tensor([[5.0000, 3.0000],
        [4.0000, 3.5000],
        [4.0000, 4.3333]])


In [69]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [71]:
xbow = torch.zeros((B,T,C)) 
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] #(t,C)
        xbow[b,t] = torch.mean(xprev,0)

In [72]:
#v2 using matmul for weighted aggregation

wei = torch.tril(torch.ones(T,T)) # lower triangular permission Table 
wei = wei / wei.sum(1, keepdims=True)
xbow2 = wei @ x #(T,T) @ (B,T,C)  Batch multiply so torch does (B,T,T) * (B,T,C) = (B,T,C)
torch.allclose(xbow,xbow2)

True

In [73]:
# v3 using softmax: Full self attentiton 
tril = torch.tril(torch.ones(T,T)) # creates a mask 
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril ==0, float('-inf'))  #future tokens cannoot commumnicate with past, we are clamping them 
wei = F.softmax(wei, dim=-1)
xbow3= wei @ x
torch.allclose(xbow2, xbow3)


True

In [74]:
# v4: using self attention
torch.manual_seed(1337)
B,T,C = 4,8,32 #Batch, Time, Channels
x= torch.randn(B,T,C)


head_size =16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias= False)
value = nn.Linear(C,head_size, bias= False)
k= key(x) # (B,T,16)
q= query(x) # (B,T, 16) 
wei = q @ k.transpose(-2,-1) #(B,T, 16) @ (B, 16,T) -> (B,T,T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril ==0, float('-inf'))
wei = F.softmax(wei,dim =-1)

v = value(x)
out = wei @ v
# out = wei @ x 
out.shape



torch.Size([4, 8, 16])

In [ ]:
#Attention is used for communication. Think of it as DAG graph each other for 

In [75]:
k= torch.randn(B,T, head_size)
q= torch.randn(B,T, head_size)
wei = q @ k.transpose(-2,-1) * head_size ** -0.5 # softmax diffusion attention
# instead of saying each token listening to all token, emphasize on particular token with greater softmax prob

torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0,2, 0.5])*8, dim=-1)

tensor([2.5045e-07, 2.2720e-08, 1.2405e-06, 1.1253e-07, 9.9999e-01, 6.1442e-06])

In [ ]:
###

In [78]:
class LayerNorm1d:   # Input->layerNorm -> attention->residual->layerNorm->feedforward -> Residual->output
    def __init__(self, dim, eps = 1e-5, momentum = 0.1):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
    
    def __call__(self,x):
        # forward pass
        xmean = x.mean(1, keepdim = True)
        xvar = x.var(1, keepdim=True)
        xhat = (x-xmean) / torch.sqrt(xvar +self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out 
    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x= torch.randn(32,100)
x = module(x)
x.shape
    

torch.Size([32, 100])